# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [46]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
import gradio as gr
from google.genai.types import Part

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('GOOGLE_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL="gemini-2.5-flash"
client = genai.Client(api_key=openai_api_key)
# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [13]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""
config = types.GenerateContentConfig(system_instruction=system_message)

In [22]:
def response(message, history):
    chat = client.chats.create(model=MODEL, config=config)

    for msg in history:
        role = msg["role"]
        text = msg["content"][0]["text"]

        if role == "user":
            chat.send_message(text)
        elif role == "assistant":
            # some SDKs support this, some don’t — safe to skip
            pass

    chat_response = chat.send_message(message)
    return chat_response.text
demo = gr.ChatInterface(response,
                            title='Chat bot',
                            textbox=gr.Textbox(placeholder="Chat"),
                            )
demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[]
[{'role': 'user', 'metadata': None, 'content': [{'text': 'helllo', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'Hello! How can FlightAI assist you today?', 'type': 'text'}], 'options': None}]
[{'role': 'user', 'metadata': None, 'content': [{'text': 'helllo', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'Hello! How can FlightAI assist you today?', 'type': 'text'}], 'options': None}, {'role': 'user', 'metadata': None, 'content': [{'text': 'My name is Hwang', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'It is a pleasure to meet you, Hwang; how may I assist you with your FlightAI travel plans today?', 'type': 'text'}], 'options': None}]
Keyboard interruption in main thread... closing server.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [48]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")

    price = ticket_prices.get(destination_city.lower())

    if price:
        return {
            "destination_city": destination_city,
            "price": price
        }
    else:
        return {
            "destination_city": destination_city,
            "price": None,
            "error": "Unknown city"
        }


In [30]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [33]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"]
    }
}

In [34]:
# And this is included in a list of tools:
tools = types.Tool(function_declarations=[price_function])
config = types.GenerateContentConfig(tools=[tools])

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def chat_fn(message, history):
    chat = client.chats.create(model=MODEL, config=config)

    # ✅ rebuild history safely
    for msg in history:
        if not msg or "role" not in msg:
            continue

        if msg["role"] == "user":
            text = msg["content"][0]["text"]
            chat.send_message(text)

    # ✅ send current message ONCE (outside loop)
    chat_response = chat.send_message(message)

    # ✅ check for function call safely
    parts = chat_response.candidates[0].content.parts

    for part in parts:
        if hasattr(part, "function_call") and part.function_call:
            fc = part.function_call

            print("Function to call:", fc.name)
            print("Args:", fc.args)

            # ✅ execute function
            if fc.name == "get_ticket_price":
                result = get_ticket_price(**fc.args)
            else:
                result = {"error": "Unknown function"}

            # ✅ ALWAYS send result back
            final_response = chat.send_message(
                Part.from_function_response(
                    name=fc.name,
                    response=result
                )
            )

            return final_response.text

    # ✅ no function call
    return chat_response.text
demo = gr.ChatInterface(chat_fn,
                            title='Chat bot',
                            textbox=gr.Textbox(placeholder="Chat"),
                            )
demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Function to call: get_ticket_price
Args: {'destination_city': 'Paris'}
Tool called for city Paris
Function to call: get_ticket_price
Args: {'destination_city': 'Paris'}
Tool called for city Paris


Traceback (most recent call last):
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\grad

Function to call: get_ticket_price
Args: {'destination_city': 'Paris'}
Tool called for city Paris


Traceback (most recent call last):
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\admin\OneDrive\Desktop\llm_engineering\.venv\Lib\site-packages\grad